In [1]:
# Load libraries and set up environment
import os 
import sys
import importlib
import datetime
import numpy as np
import pandas as pd
import anndata as ad    
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import gffutils
import anndata as ad
import scanpy as sc 
from tqdm import tqdm

# Ensure CUDA is available and if not use CPU
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device found")

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

float_type = {"device": device, "dtype": torch.float}
if device.type == "cuda":
    torch.set_default_tensor_type(torch.cuda.FloatTensor)

# Set seed for reproducibility
torch.manual_seed(0)

# Configure plotting styles
sns.set_theme()
sc.set_figure_params(figsize=(7, 7), frameon=True, dpi=80, facecolor='white')

# Define module paths
src_path = "/gpfs/commons/home/kisaev/Leaflet-private/src/"

# Add to sys.path if not already present
if src_path not in sys.path:
    sys.path.append(src_path)

# Import custom modules
import BetaDirichletFactor.LeafletFA as LeafletFA
import BetaDirichletFactor.LeafletFAminibatch as LeafletFAminibatch
import BetaDirichletFactor.differential_splicing as ds
import BetaDirichletFactor.utils as utils
import visualization.IsovizPy as ja
import evaluations.cost_correlation_assign as cca

# Reload custom modules
importlib.reload(LeafletFA)
importlib.reload(ds)
importlib.reload(utils)

# Simulation source code
sys.path.append("/gpfs/commons/home/kisaev/Leaflet-private/src/simulation/")
import simulate_counts as sim 
importlib.reload(sim)


Torch version: 2.4.1.post300
CUDA available: True
CUDA device count: 3
CUDA device name: NVIDIA L40S
Using device: cuda


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1728241823685/work/torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)


Torch Version: 2.4.1.post300
CUDA Version: 12.0
Torch Version: 2.4.1.post300
CUDA Version: 12.0
Torch Version: 2.4.1.post300
CUDA Version: 12.0


<module 'simulate_counts' from '/gpfs/commons/home/kisaev/Leaflet-private/src/simulation/simulate_counts.py'>

In [2]:
# Define base output directory
import json 
base_output_dir = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/Simulations/2025/manuscript_sim_analysis/2025-02-19"

# Get parameter set ID from command line
# param_id = int(sys.argv[1])
param_id = 0 

# Load parameters
param_file = os.path.join(base_output_dir, "parameter_combinations.json")
with open(param_file, "r") as f:
    param_list = json.load(f)
params = param_list[param_id] 

# Define output directory
output_dir = os.path.join(base_output_dir, f"run_{param_id}")
os.makedirs(output_dir, exist_ok=True)
print(f"All outputs will be saved in {output_dir}")

All outputs will be saved in /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/Simulations/2025/manuscript_sim_analysis/2025-02-19/run_0


In [3]:
# Anndata file input file path 
ATSE_anndata_file="/gpfs/commons/groups/knowles_lab/Karin/TMS_MODELING/DATA_FILES/BRAIN_ONLY/02112025/TMS_Anndata_ATSE_counts_with_waypoints_20250211_171237.h5ad"
ATSE_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/ATSEmap/output/ATSEfiles/TMS_atse_file_unanno_also_2025-01-30_19-24-18.txt.gz"

In [4]:
# Load splicing anndata file along with the ATSE annotation file (both obtained using upstream processing within LeafletFA framework)
adata = ad.read_h5ad(ATSE_anndata_file)
atses = pd.read_csv(ATSE_file, sep="\t")

### Simulate data!

In [5]:
# Filter adata to only include junctions that have non_zero_count_cells >= 10
adata = adata[:, adata.var["non_zero_count_cells"] > 2]

# choose which column should be used for maintaining cell labels when simulating data...
sim_label_column = params["sim_label_column"] #"cell_type_grouped" # or set to None then cells will be randomly assigned to groups
proportion_negative = params["proportion_negative"]

if params["sim_label_column"] is None:
    K = 2
else:
    K = len(adata.obs[sim_label_column].unique())

# Set up some useful params 
params["input_conc"] = None if params["input_conc"] is None else torch.tensor(np.inf)
input_conc = params["input_conc"]
junc_specific_prior = params["junc_specific_prior"] # set to True if you want to use a junc-specific prior (a set of a,b shape params for each junction) or False to learn one set of a,b shape params for all junctions
waypoints_use = params["waypoints_use"] # don't have waypoints in simulated data

In [6]:
sim_label_column = "cell_type_grouped"
K = len(adata.obs[sim_label_column].unique())
proportion_negative = 0.5
print(K)

9


In [7]:
# Preprocess the data
adata_filtered = sim.preprocess_adata(adata, sim_label_column, "cell_by_cluster_matrix")
# Simulate data
_, _, adata_input, cell_type_psi_df = sim.simulate_and_prepare_data(adata_filtered, K, float_type, proportion_negative, sim_label_column)

# Write cell_type_psi_df to a CSV file 
cell_type_psi_df_path = os.path.join(output_dir, 'cell_type_psi_df.csv')
cell_type_psi_df.to_csv(cell_type_psi_df_path, index=False)

Filtering ATSEs to remove those very unevenly distributed across cell types!


100%|██████████| 9/9 [00:30<00:00,  3.44s/it]
/gpfs/commons/home/kisaev/Leaflet-private/src/simulation/simulate_counts.py:281: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cluster_junc_counts =  adata_input.var.groupby(["event_id"]).agg({"junction_id": "count"}).reset_index()
/gpfs/commons/home/kisaev/Leaflet-private/src/simulation/simulate_counts.py:307: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for cluster, juncs_c in tqdm( adata_input.var.groupby("event_id")):


Cluster_Counts nnz: 14507235
Junction_Counts nnz: 8005369
The number of unique junctions included in the simulation data is: 10170
The number of unique clusters included in the simulation data is: 3390


100%|██████████| 3390/3390 [00:02<00:00, 1220.74it/s]
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Cluster_Counts nnz: 14004504
Junction_Counts nnz: 7578659
The proportion of negative ASEs to set is: 0.5
The number of cell types is: 9
The number of cells is: 19942
The number of junctions is: 9798
Number of negative labels (0): 1633
Number of positive labels (1): 1633


...

Done normalizing junction counts by cluster!
Done simulating junction counts!
True label counts:
 true_label
positive    4899
negative    4899
Name: count, dtype: int64
Sample label counts:
 sample_label
positive    4899
negative    4899
Name: count, dtype: int64


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/storage.py:39: ImplicitModificationWarning: Layer 'junc_ratio' should not be a np.matrix, use np.ndarray instead.
  warnings.warn(msg, ImplicitModificationWarning)
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/storage.py:48: FutureWarning: AnnData previously had undefined behavior around matrices of type <class 'scipy.sparse._coo.coo_matrix'>.In 0.12, passing in this type will throw an error. Please convert to a supported type.Continue using for this minor version at your own risk.
  warnings.warn(msg, FutureWarning)


Cluster_Counts nnz: 13836771
Junction_Counts nnz: 11859267


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/storage.py:48: FutureWarning: AnnData previously had undefined behavior around matrices of type <class 'scipy.sparse._coo.coo_matrix'>.In 0.12, passing in this type will throw an error. Please convert to a supported type.Continue using for this minor version at your own risk.
  warnings.warn(msg, FutureWarning)
/gpfs/commons/home/kisaev/Leaflet-private/src/simulation/simulate_counts.py:405: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1728241823685/work/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  ).to_sparse_csr()


Data successfully simulated and prepared!


In [8]:
# Check column names that are not strings
problematic_columns = [col for col in adata_input.var.columns if not isinstance(col, str)]
print("Problematic columns:", problematic_columns)

# Convert problematic column names to strings
adata_input.var.columns = adata_input.var.columns.map(str)

if adata_input.var.columns.duplicated().any():
    print("Duplicate column names found. Resolving by renaming.")
    adata_input.var.columns = pd.io.parsers.ParserBase({'names': adata_input.var.columns})._maybe_dedup_names(adata_input.var.columns)

import scipy 
from scipy.sparse import csr_matrix

# Check if any layers are in COO format
for layer_name, layer_data in adata_input.layers.items():
    if isinstance(layer_data, scipy.sparse.coo_matrix):
        print(f"Converting {layer_name} from COO to CSR format.")
        adata_input.layers[layer_name] = layer_data.tocsr()

# Convert adata.X if it is in COO format
if isinstance(adata_input.X, scipy.sparse.coo_matrix):
    print("Converting adata.X from COO to CSR format.")
    adata_input.X = adata_input.X.tocsr()

# Convert varm, obsm, and obsp if they contain COO matrices
for attr in ['varm', 'obsm', 'obsp']:
    for key in getattr(adata_input, attr).keys():
        if isinstance(getattr(adata_input, attr)[key], scipy.sparse.coo_matrix):
            print(f"Converting {attr}['{key}'] from COO to CSR format.")
            getattr(adata_input, attr)[key] = getattr(adata_input, attr)[key].tocsr()


Problematic columns: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Converting cell_by_junction_matrix from COO to CSR format.


Converting cell_by_cluster_matrix from COO to CSR format.


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/storage.py:48: FutureWarning: AnnData previously had undefined behavior around matrices of type <class 'scipy.sparse._coo.coo_matrix'>.In 0.12, passing in this type will throw an error. Please convert to a supported type.Continue using for this minor version at your own risk.
  warnings.warn(msg, FutureWarning)


In [9]:
# save adata_input to h5ad file
output_anndata = "/gpfs/commons/groups/knowles_lab/Karin/TMS_MODELING/DATA_FILES/SIMULATED"
# get todays date
today = datetime.datetime.now()
today = today.strftime("%Y-%m-%d")
adata_input.write_h5ad(f"{output_anndata}/simulated_data_{today}.h5ad")
print(f"Simulated anndata saved to {output_anndata}/simulated_data_{today}.h5ad")

... storing 'event_id' as categorical
... storing 'chr' as categorical
... storing 'start' as categorical
... storing 'end' as categorical
... storing 'sample_label' as categorical
... storing 'true_label' as categorical


Simulated anndata saved to /gpfs/commons/groups/knowles_lab/Karin/TMS_MODELING/DATA_FILES/SIMULATED/simulated_data_2025-03-06.h5ad


### Initialize and train the model using the simulated counts!

In [10]:
num_inits = params["num_inits"]
num_epochs = params["num_epochs"]
num_samples = params["num_samples"]
lr = params["lr"]
ELBO_num_particles = params["ELBO_num_particles"]
print_epochs = 10

In [11]:
lr = 0.8
num_inits = 5

In [12]:
# Let's initialize the LeafletFA class 
importlib.reload(LeafletFA)
leaflet_model = LeafletFA.LeafletFA(adata=adata_input, K=K, 
                                    junc_specific_prior = junc_specific_prior, 
                                    waypoints_use=waypoints_use, 
                           input_conc_prior = input_conc, 
                           num_epochs=200, 
                           print_epochs=print_epochs, 
                           ELBO_num_particles=5, 
                           lr=lr, gamma=0.5, 
                           num_samples=100)

# Convert AnnData into PyTorch tensors for model training 
leaflet_model.from_anndata()

Torch Version: 2.4.1.post300
CUDA Version: 12.0
Taking in the AnnData object with 19942 cells and 9798 junctions.
Processing AnnData on cuda


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/storage.py:48: FutureWarning: AnnData previously had undefined behavior around matrices of type <class 'scipy.sparse._coo.coo_matrix'>.In 0.12, passing in this type will throw an error. Please convert to a supported type.Continue using for this minor version at your own risk.
  warnings.warn(msg, FutureWarning)


In [13]:
leaflet_model.y, leaflet_model.total_counts

(tensor(indices=tensor([[    0,     0,     0,  ..., 19941, 19941, 19941],
                        [    0,     1,     2,  ...,  6840,  6841,  6842]]),
        values=tensor([13.,  3., 18.,  ...,  0.,  1.,  0.]),
        size=(19942, 9798), nnz=14004504, layout=torch.sparse_coo),
 tensor(indices=tensor([[    0,     0,     0,  ..., 19941, 19941, 19941],
                        [    0,     1,     2,  ...,  6840,  6841,  6842]]),
        values=tensor([34., 34., 34.,  ...,  1.,  1.,  1.]),
        size=(19942, 9798), nnz=14004504, layout=torch.sparse_coo))

In [14]:
# Train the model 
leaflet_model.train(num_initializations=num_inits)

# Get the best initialization and extract all the latent variables at this initialization
# If you want the latent variables from a different initialization, you can pass the index of that initialization to the get_all_variables() function
leaflet_model.get_all_variables()

Random seeds: [9784, 1059, 9452, 6499, 7998]
Training LeafletFA with 5 initializations.
Input concentration prior: None
Junction-specific prior: True
Initial K to learn: 9
Random initialization of variational parameters!
-------------------------------------------------
Initialization #1 with seed 9784
-------------------------------------------------
Training in progress for 200 epochs!
Epoch 0: Loss = 114173810.71371299
Epoch 10: Loss = 74389715.92119285
Epoch 20: Loss = 61074928.13947017
Epoch 30: Loss = 57090499.85016167
Epoch 40: Loss = 54495195.029574
Epoch 50: Loss = 52772038.88959491
Epoch 60: Loss = 51316230.431331925
Epoch 70: Loss = 50412775.3491137
Epoch 80: Loss = 49512872.548098564
Epoch 90: Loss = 48514329.30835095
Epoch 100: Loss = 48034428.15037463
Epoch 110: Loss = 47454196.83082546
Epoch 120: Loss = 46914958.76437558
Epoch 130: Loss = 46448203.93428473
Epoch 140: Loss = 45927053.30090912
Epoch 150: Loss = 45406305.0978778
Epoch 160: Loss = 45014920.790507995
Epoch 17

In [ ]:
# let's look at the results 
assign_matrices = [result["summary_stats"]["assign"]["mean"] for result in leaflet_model.latent_results]

# Calculate correlations between initializations if more than 2 
if num_inits > 1:
    avg_corr, median_corr, min_corr = utils.calculate_and_plot_correlations(assign_matrices)
else: 
    avg_corr, median_corr, min_corr = None, None, None  # Set default values when there's only one initialization

In [ ]:
# Prune K: note this updates all the latent variables in the model to only include estimates for the pruned K
leaflet_model.prune_K()

In [ ]:
LEAFLETFA_LATENT_KEY = "X_leafletFA"
adata_input.obsm[LEAFLETFA_LATENT_KEY] = leaflet_model.assign_post # assign_post is the posterior assignment cell-factor activity matrix 
sc.pp.neighbors(adata_input, use_rep=LEAFLETFA_LATENT_KEY, n_neighbors=10)

In [ ]:
# sc.tl.umap(adata_input)
# sc.pl.umap(adata_input, color=["cell_type"])

In [ ]:
cell_tye_silhouette = ds.calculate_silhouette_score(leaflet_model.assign_post, adata_input.obs.cell_type.values)
print(f"Silhouette score for cell types: {cell_tye_silhouette}")

In [ ]:
pi_df = pd.DataFrame(leaflet_model.pi, columns=["factor_assignment_probabilities"])
pi_df["factor_K"] = pi_df.index

# Sort by factor_assignment_probabilities in descending order
pi_df = pi_df.sort_values(by="factor_assignment_probabilities", ascending=False)

# Make sorted barplot
plt.figure(figsize=(10, 5))
sns.barplot(x="factor_K", y="factor_assignment_probabilities", data=pi_df, order=pi_df["factor_K"])
plt.title("Factor K Probabilities (Sorted)")
plt.xlabel("Factor K")
plt.ylabel("Assignment Probabilities")
plt.xticks(rotation=90)  # Rotate x-axis labels if many factors
plt.show()

In [ ]:
# Let's extract sampled PSI means to calculate imputed difference between groups
# convert leaflet_model.psi_samples to a dataframe rename columns to factor_
factor_psi_df = pd.DataFrame(leaflet_model.psi_learned.T)
factor_psi_df.columns = [f"factor_imputed_psi_{col}" for col in factor_psi_df.columns]

if K == 2:
    factor_psi_df["imputed_diff"] = np.abs(factor_psi_df["factor_imputed_psi_0"] - factor_psi_df["factor_imputed_psi_1"])
if K > 2:
    # Calculate variance across all factors
    factor_psi_df["imputed_diff"] = factor_psi_df.var(axis=1)

# Add factor_psi_df to adata_input.var 
adata_input.var = pd.concat([adata_input.var, factor_psi_df], axis=1)

# Plot scatterplot correlation between imputed difference and true difference 
plt.figure(figsize=(6, 6))
sns.scatterplot(x="imputed_diff", y="difference", data=adata_input.var)
plt.xlabel("Imputed Difference")
plt.ylabel("True Difference")
plt.title("Imputed vs True Difference in PSI")
plt.show()

# Calculate pearson and spearman correlation
pearson_corr = stats.pearsonr(adata_input.var["imputed_diff"], adata_input.var["difference"])[0]
spearman_corr = stats.spearmanr(adata_input.var["imputed_diff"], adata_input.var["difference"])[0]
print(f"Spearman correlation between imputed and true difference: {spearman_corr}")

In [ ]:
if K > 2:
    results = ds.analyze_all_factors(leaflet_model, fdr_threshold=0.05, min_effect_size=0.1)

    # Let's collect all the dataframes from results
    # go through results.keys() and collect second value in tuple and add column indicating factor K
    results_df = pd.DataFrame()
    for key in results.keys():
        factor_res = results[key][1]
        factor_res["factor_K"] = key
        results_df = pd.concat([results_df, factor_res])

    # for every unique junctions i want to summarize number of factors that it was significant in
    sig_only = results_df[results_df["significant"] == True]
    sig_only = sig_only.groupby("junction_idx").agg({"significant": "count"}).reset_index()
    sig_only = sig_only.rename(columns={"significant": "num_factors_significant"})
    # merge with results_df
    results_df = results_df.merge(sig_only, on="junction_idx", how="left")
    results_df["abs_effect_size"] = np.abs(results_df["effect_size"])

    # plot dist of effect sizes
    plt.figure(figsize=(6, 6))
    sns.histplot(results_df["effect_size"], bins=30)
    plt.xlabel("Effect Size")
    plt.ylabel("Frequency")
    # save to output_dir
    plt.savefig(os.path.join(output_dir, "effect_size_dist.png"))

In [ ]:
junc_idx = 0 
psi_samples_df = pd.DataFrame(leaflet_model.psi_samples[:, :, junc_idx])
# Add "factor_" prefix to all columns
psi_samples_df.columns = [f"factor_{col}" for col in psi_samples_df.columns]
# Draw violin plot for each factor ylim 0,1 show median 
plt.figure(figsize=(10, 5))
sns.boxplot(data=psi_samples_df)
plt.ylim(-0.1, 1.1)
plt.title(f"PSI samples for junction {junc_idx}")
# reduce font size of x-axis labels and rotate a little
plt.xticks(fontsize=8, rotation=45)
plt.show()

In [ ]:
importlib.reload(ds)

In [ ]:
if K == 2:
    results_dict, results_df = ds.analyze_differential_splicing(leaflet_model, factor_idx = 0, 
                                                            fdr_threshold=0.05, 
                                                            min_effect_size=0.1)
                                        
    # Add results_df to adata_input.var
    adata_input.var = pd.concat([adata_input.var, results_df], axis=1)

    # Relabel positive true_label junctions as negative if their difference is less than 0.15 (not doing atm)
    adata_input.var["adjusted_true_label"] = adata_input.var["true_label"]

    # Create confusion matrix directly from DataFrame
    conf_matrix = pd.crosstab(adata_input.var['adjusted_true_label'], adata_input.var['significant'], 
                             normalize=False)  # set to True if you want percentages

    # Plot
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted (significant)')
    plt.ylabel('True Label')
    plt.show()

    # add absolute effective size 
    adata_input.var["abs_effect_size"] = np.abs(adata_input.var["effect_size"])
    # Plot scatterplot correlation between imputed difference and true difference 
    plt.figure(figsize=(6, 6))
    sns.scatterplot(x="abs_effect_size", y="difference", data=adata_input.var)
    plt.xlabel("DS Effect Size")
    plt.ylabel("True Estimated Difference")
    plt.title("Imputed vs True Difference in PSI")
    plt.show()

    # Calculate pearson and spearman correlation
    pearson_corr = stats.pearsonr(adata_input.var["abs_effect_size"], adata_input.var["difference"])[0]
    spearman_corr = stats.spearmanr(adata_input.var["abs_effect_size"], adata_input.var["difference"])[0]
    print(f"Pearson correlation between abs_effect_size and true difference: {pearson_corr}")
    print(f"Spearman correlation between abs_effect_size and true difference: {spearman_corr}")

In [ ]:
true_positive_junctions = set(adata_input.var[adata_input.var["adjusted_true_label"] == "positive"].index)
df_fdr_results = ds.calibration_test(leaflet_model, true_positive_junctions, min_effect_size=0.1)

In [ ]:
# Example usage
precision, recall, auc_pr = ds.plot_precision_recall_curve(adata_input, results_df)
fpr, tpr, auc_roc = ds.plot_roc_curve(adata_input, results_df)

In [ ]:
# color by adjusted_true_label
plt.figure(figsize=(6, 6))
sns.histplot(data=adata_input.var, x="effect_size", hue="adjusted_true_label")
xlabel = "Estimated Effect Size B_j"
ylabel = "Frequency"
plt.xlabel(xlabel)
plt.ylabel(ylabel)

In [ ]:
# Collect all params used in this analysis as well as the results

# Create a dataframe to store results
results_df = pd.DataFrame([{
    "param_id": param_id,
    "K": K,
    "junc_specific_prior": params["junc_specific_prior"],
    "input_conc": "inf" if params["input_conc"] is not None else "None",
    "num_epochs": params["num_epochs"],
    "num_inits": params["num_inits"],
    "cell_type_silhouette": cell_tye_silhouette,
    "avg_corr": avg_corr,
    "median_corr": median_corr,
    "min_corr": min_corr,
    "lr": params["lr"],
    "ELBO_num_particles": params["ELBO_num_particles"],
    "num_samples": params["num_samples"],
    "proportion_negative": params["proportion_negative"],
    "pruned_K": len(leaflet_model.pi)  # Number of factors retained after pruning
}])

# Save results dataframe
results_file = os.path.join(output_dir, "run_summary.csv")
results_df.to_csv(results_file, index=False)

print(f"Saved run summary to {results_file}")

In [ ]:
leaflet_model.y = None
leaflet_model.total_counts = None

In [ ]:
leaflet_model.adata = None


In [ ]:
leaflet_model.all_params